# Imports

In [ ]:
from Functions.Fcts_DimRed import select_dimred_features, compute_PCA, compute_DC, compute_UMAP, compute_tSNE, compute_kmeans, compute_phenograph, run_slingshot, subsample_anndata_geosketch, random_subset_anndata_frac, random_subset_anndata, knn_label_transfer
from Functions.Fcts_Plotting import plot_random_organoids_per_cluster
from Functions.Fcts_Base import load_adata
import seaborn as sns

%load_ext autoreload
%autoreload 2

# User input

In [ ]:
load_dir = "PATH TO OUTPUT FROM 3_QualityControl" # Path to .h5ad file containing the AnnData object.

remove_feats_dimred = [] # List of str. Features which contain these strings are removed for the computation.

# Set up plotting parameters
fig_param = {"axes.spines.right": False,
                  "axes.spines.top": False,
                  "axes.spines.left": True,
                  "axes.spines.bottom": True,
                  "pdf.fonttype": 42,
                  "font.size": 6,
                  "axes.labelsize": 6,
                  "axes.titlesize": 8,
                  "xtick.labelsize": 6,
                  "ytick.labelsize": 6,
                  "legend.fontsize": 6}

sns.set_theme(rc = fig_param, style = "ticks")

# Load & misc
Loads AnnData object and sets up an AnnData object for dimensionality reduction based on common features across all organoids not part of remove_features

In [ ]:
# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad = load_adata(load_dir)
dimred_feats = select_dimred_features(ad, remove_feats_dimred)

# PCA

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
plot_feat = ["R0__DAPI_mean", "area", "Medium"] # Features to plot on PCA using hue.
number_components = 5 # Number of PCA components to compute.
representation = "z_scaled" # Key for .layers (expression matrix) or .obsm (representation, e.g. "X_pca"). If None, uses ad.X.
save_plot = False

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad = compute_PCA(
    ad,
    plot_feat, 
    number_components = number_components, 
    dimred_feats = dimred_feats, 
    representation = representation, 
    palette = "inferno", 
    save_plot = save_plot
    )

# DC

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
plot_feat = ["R0__DAPI_mean", "area", "Medium"]
number_components = 10 # Number of DC to compute.
representation = None # Key for .layers (expression matrix) or .obsm (representation, e.g. "X_pca"). If None, uses ad.X.
n_neighbors = 50 # Number of neighbors for the diffusion components computation.
save_plot = False

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad = compute_DC(
    ad,
    plot_feat, 
    number_components = number_components, 
    dimred_feats=dimred_feats, 
    representation = representation, 
    palette = "inferno", 
    save_plot = save_plot, 
    n_neighbors = n_neighbors
    )

# UMAP

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
plot_feat = ["R0__DAPI_mean", "area", "Medium"]
n_neighbors = 10 # Number of neighbors for UMAP.
min_dist = [0.1] # Minimum distances for UMAP.
representation = "X_pca" # Key for .layers (expression matrix) or .obsm (representation, e.g. "X_pca"). If None, uses ad.X.
save_plot = False

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad = compute_UMAP(ad,
    plot_feat = plot_feat, 
    n_neighbors = n_neighbors, 
    min_dist = min_dist, 
    dimred_feats = dimred_feats, 
    save_plot = save_plot, 
    representation=representation
    )

# tSNE

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
plot_feat = ["Medium"] # Features to plot in the tSNE. Can be a list of features or a single feature.
perplexity = [20, 2] # Perplexity for tSNE. Can be a list of perplexities to compute multiple tSNEs.
representation = None # Key for .layers (expression matrix) or .obsm (representation, e.g. "X_pca"). If None, uses ad.X (raw data).
save_plot = False

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad = compute_tSNE(
    ad,
    plot_feat = plot_feat,
    perplexity = perplexity,
    dimred_feats = dimred_feats,
    save_plot = save_plot,
    representation = representation,
    )

# K-Means Clustering

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
n_clusters = [3, 5, 6] # Number of clusters to compute.
representation = "X_pca" # Key for .layers (expression matrix) or .obsm (representation, e.g. "X_pca"). If None, uses ad.X.
visualize_on = "X_umap"  # e.g., "X_umap", "X_tsne", "X_pca", or None to auto-pick
save_plot = False

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad = compute_kmeans(
    ad,
    n_clusters = n_clusters,
    dimred_feats = dimred_feats,
    save_plot = save_plot,
    representation = representation,
    visualize_on = visualize_on,     # e.g., "X_umap", "X_tsne", "X_pca", or None to auto-pick
    point_size = 6)

# PhenoGraph

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
k = 100 # Number of neighbors for PhenoGraph. Can be a list of k values to compute multiple PhenoGraphs.
representation = None # Key for .layers (expression matrix) or .obsm (representation, e.g. "X_pca"). If None, uses ad.X.
visualize_on = "X_umap"  # e.g., "X_umap", "X_tsne", "X_pca", or None to auto-pick
save_plot = False

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad = compute_phenograph(
    ad,
    k = k,
    dimred_feats = dimred_feats,
    save_plot = save_plot,
    representation = representation,
    visualize_on = visualize_on,
    point_size = 6)

# Plot organoids

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
group_by = "Medium" # Group by this key in ad.obs.
n = 8 # Number of organoids to plot per cluster.
stainings = ["R0__DAPI", "R1__DAPI"] # List of stainings to plot.
thresholds = [None, None] # Thresholds for the stainings. Can be a list of thresholds for each staining. None leads to no thresholding.
normalize_sizes = True  # Adjusts organoid sizes to the largest organoid if True.
bar_length_um = None  # desired scalebar length in microns; if None, auto-pick a “nice” value
save_plot = False

# ── Run Cell ───────────────────────────────────────────────────────────────────────
fig = plot_random_organoids_per_cluster(
    ad,
    cluster_key = group_by,
    n_per_cluster = n,
    stainings = stainings,
    pyramid_level = 0,
    seed = 0,
    cmap="magma",
    save_plot = save_plot,
    thresholds = thresholds,
    normalize_sizes = normalize_sizes,
    bar_length_um = bar_length_um,
    add_boundary = False, 
    )

# SlingShot

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
start_cluster = "5" # Cluster to start the lineage inference from (str or int).
save_plot = False
cluster_key = "kmeans_labels" # Column in ad.obs with cluster labels (categorical or int/str).
visualize_on = "X_umap"  # e.g., "X_umap",

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad = run_slingshot(
ad,
start_cluster = start_cluster,
num_epochs = 500,
save_plot = save_plot,
cluster_key = cluster_key,
visualize_on = visualize_on,
)

# Helper Functions

### Geometric Sketching

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
fraction = 0.5 # Fraction of observations to sample (0 < fraction <= 1).
use_rep = "X_pca" # Key for .layers (expression matrix) or .obsm (representation, e.g. "X_pca") to use for geosketch. If None, uses ad.X (raw data).

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad_subsampled_geo = subsample_anndata_geosketch(ad, fraction=fraction, use_rep=use_rep, random_state=0)

### Pick random objects

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
fraction = 0.5 # Fraction of observations to sample (0 < fraction <= 1).

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad_subset_frac = random_subset_anndata_frac(ad,
    fraction = fraction, 
    random_state = 0)

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
n = 25 # Number of observations to sample

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad_subset = random_subset_anndata(ad,
    n = n, 
    random_state = 0)

### Upscale obs from Sketched to full data

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
obs = "phenograph_labels" # Column in ad_subset.obs with labels to transfer.
n_neighbors = 15 # Number of neighbors to consider for the kNN classifier.
obsm_key = None # Name of a matrix in .obsm to use as features (e.g., "X_pca"). If provided, used for both reference and target. Mutually exclusive with layer.
layer = None # Optional name of a numeric layer to use instead of .X. If provided, used for both reference and target.

# ── Run Cell ───────────────────────────────────────────────────────────────────────
knn_label_transfer(
    adata_ref = ad_subset,
    adata_target = ad,
    label_key = obs,
    n_neighbors = n_neighbors,
    obsm_key = obsm_key,         
    layer = layer,      
    features = dimred_feats,    
    target_obs_key = obs,
    copy=False
)